# Train the DeepDarts YOLOv8n detection model

Reproduces the model bundled at `cv/src/main/assets/models/deepdarts-yolov8.onnx` (spec
`014-dnn-dart-detection`). Trains a YOLOv8n object-detection model on the public **DeepDarts**
dataset (Roboflow Universe, `testing-zzmc9/deepdarts-yolov8`, CC BY 4.0 — see `NOTICE.md`) to
detect 5 classes: `0` = dart tip, `1..4` = board calibration points (`cal_1..cal_4`).

**Not part of the app build.** This is a one-time (or re-run-when-retraining) offline step —
its only output the app cares about is the final `.onnx` file, copied into
`cv/src/main/assets/models/`. See `specs/014-dnn-dart-detection/research.md` and `quickstart.md`
step 0 for the design rationale (why opset 12, why `nms=False`, why YOLOv8n over YOLOv8s, etc).

**Requirements**: a Roboflow account (free tier is enough) and its API key — get yours at
https://app.roboflow.com → Account → Roboflow Keys. A CUDA GPU is strongly recommended (this
took ~4 minutes per 100 epochs on an RTX 4070 Super; CPU-only will be much slower) but not
required — Ultralytics falls back to CPU automatically.

## 1. Install dependencies

`ultralytics` (YOLOv8 training/export) and `roboflow` (dataset download). If ONNX export later
fails with `ModuleNotFoundError: No module named 'onnx'`, also run:
`pip install --user "onnx>=1.12.0,<2.0.0" onnxruntime "onnxslim>=0.1.82"` — Ultralytics normally
auto-installs these on first ONNX export, but that auto-install can fail silently in locked-down
Python environments (e.g. the Microsoft Store Python package on Windows, which can't write to
its own site-packages).

In [ ]:
%pip install ultralytics roboflow

## 2. Verify GPU availability (optional but recommended)

In [ ]:
import torch
import ultralytics

print("torch", torch.__version__, "cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
print("ultralytics", ultralytics.__version__)

## 3. Download the DeepDarts YOLOv8 dataset from Roboflow

Set `ROBOFLOW_API_KEY` below (or as an environment variable) to your own key — **never commit a
real key to this notebook**. The dataset is public (CC BY 4.0) but Roboflow still requires an
authenticated request to download it; there is no anonymous/public bypass (confirmed while
building this feature — an invalid/placeholder key gets a clean 401, not a downgrade to
anonymous access).

In [ ]:
import os

# Prefer an environment variable over hardcoding your key here.
ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY", "")
assert ROBOFLOW_API_KEY, "Set ROBOFLOW_API_KEY (env var or above) to your Roboflow API key"

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("testing-zzmc9").project("deepdarts-yolov8")
print(project)
print("available versions:", [v.version for v in project.versions()])

In [ ]:
# Version 3 was the latest at the time this model was trained (1,178 image files across
# train/valid/test after Roboflow's augmentation, ~583 unique source photos). Pin the version
# explicitly so a retrain is reproducible even if the dataset gets a new version later.
dataset = project.version(3).download("yolov8", location="DeepDarts-YOLOv8-3")
print("downloaded to", dataset.location)

In [ ]:
# Sanity-check the dataset structure and class list.
import yaml

with open(f"{dataset.location}/data.yaml") as f:
    data_yaml = yaml.safe_load(f)
print(data_yaml)

## 4. Train YOLOv8n

The shipped model used 300 epochs with light augmentation (small rotation/translation/scale
jitter) — an initial 100-epoch run without augmentation got mAP50 0.894 overall but only 0.486
on the dart-tip class specifically (recall 0.46, i.e. missed over half the darts); the 300-epoch
run below improved that to mAP50 0.91 overall / 0.571 on dart-tip (recall 0.49). Dart-tip
detection remains the weaker class — small object, motion blur/occlusion in the source photos —
documented as a known limitation in `CLAUDE.md`'s Known Gaps section, not something this training
recipe alone fixes. `patience=50` early-stops if validation mAP plateaus.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=300,
    imgsz=640,
    batch=16,
    project="runs",
    name="deepdarts-yolov8n",
    patience=50,
    degrees=5,
    translate=0.05,
    scale=0.3,
)

## 5. Check validation metrics

Expect roughly: calibration-point classes (`1..4`) at mAP50 ≈ 0.99+ (they're large, high-contrast,
always-present markers — easy for the model); dart tip (`0`) noticeably lower (small, occluded,
variable-count object). If dart-tip recall regresses well below ~0.4, something likely went wrong
with augmentation strength or epoch count — compare against the numbers in the markdown cell
above before shipping a new `.onnx`.

In [ ]:
metrics = model.val()
print(metrics)

## 6. Export to ONNX

`opset=12`, `simplify=True`, `nms=False`, static `imgsz=640` — this exact combination is what
`cv/opencv/dnn/YoloV8Model.kt` and `YoloV8OutputDecoder.kt` are written against
(`research.md` §1): OpenCV's `dnn` ONNX importer needs a static input shape and no embedded NMS
layer (NMS is done in Kotlin instead, so its behavior doesn't depend on OpenCV-version quirks).
Changing any of these flags requires updating the Kotlin side to match — the output tensor shape
`(1, 4 + numClasses, numAnchors)` in particular is hardcoded there.

In [ ]:
best_weights = model.trainer.best  # path to runs/detect/deepdarts-yolov8n/weights/best.pt
export_model = YOLO(best_weights)
onnx_path = export_model.export(format="onnx", opset=12, simplify=True, nms=False, imgsz=640)
print("exported to", onnx_path)

## 7. Verify the export loads via OpenCV's `dnn` module

Same check the Android app implicitly relies on (`Dnn.readNetFromONNX`) — run it here first so a
broken export is caught before it's copied into the app. Expect output shape `(1, 9, 8400)`:
9 = 4 box coords + 5 classes, 8400 = anchor count for a 640x640 input with YOLOv8's default
stride set. `pip install opencv-python-headless` if `cv2` isn't already available.

In [ ]:
import cv2
import numpy as np

net = cv2.dnn.readNetFromONNX(str(onnx_path))
blob = np.random.rand(1, 3, 640, 640).astype(np.float32)
net.setInput(blob)
out = net.forward()
print("opencv version", cv2.__version__)
print("output shape:", out.shape, out.dtype)
assert out.shape == (1, 9, 8400), "unexpected output shape -- check numClasses/imgsz match YoloV8Model.kt"

## 8. Copy into the app

Replaces the bundled model asset. After this, rebuild/retest the `cv` module
(`./gradlew :cv:test`, and the instrumented tests on a device) before committing.

In [ ]:
import shutil
from pathlib import Path

# Adjust if this notebook is run from somewhere other than the repo's scripts/ directory.
repo_root = Path.cwd().parent
dest = repo_root / "cv" / "src" / "main" / "assets" / "models" / "deepdarts-yolov8.onnx"
dest.parent.mkdir(parents=True, exist_ok=True)
shutil.copy(onnx_path, dest)
print("copied to", dest)

## 9. End-to-end sanity check (optional)

Runs the exact same letterbox → forward → decode → calibration pipeline `YoloV8Model.kt` /
`OpenCvDnnBoardDetector.kt` / `CalibrationPointMapper.kt` implement, in Python, against one real
validation image — useful for catching a bad retrain (e.g. all 4 calibration points not detected,
wildly wrong rotation) before it ever reaches an Android build.

In [ ]:
import glob
import math

img_path = glob.glob(f"{dataset.location}/valid/images/*.jpg")[0]
img = cv2.imread(img_path)
h0, w0 = img.shape[:2]
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

size = 640
scale = min(size / w0, size / h0)
nw, nh = int(w0 * scale), int(h0 * scale)
resized = cv2.resize(img_rgb, (nw, nh))
padx, pady = (size - nw) // 2, (size - nh) // 2
canvas = np.full((size, size, 3), 114, dtype=np.uint8)
canvas[pady : pady + nh, padx : padx + nw] = resized

blob = cv2.dnn.blobFromImage(canvas, 1 / 255.0, (size, size), (0, 0, 0), swapRB=False, crop=False)
net.setInput(blob)
out = net.forward().reshape(9, 8400)

conf_thr, iou_thr = 0.25, 0.45
dets = []
for a in range(out.shape[1]):
    box = out[0:4, a]
    scores = out[4:9, a]
    cls = int(np.argmax(scores))
    conf = scores[cls]
    if conf >= conf_thr:
        dets.append((cls, box[0], box[1], box[2], box[3], conf))


def iou(a, b):
    ax1, ay1, ax2, ay2 = a[1] - a[3] / 2, a[2] - a[4] / 2, a[1] + a[3] / 2, a[2] + a[4] / 2
    bx1, by1, bx2, by2 = b[1] - b[3] / 2, b[2] - b[4] / 2, b[1] + b[3] / 2, b[2] + b[4] / 2
    ix1, iy1, ix2, iy2 = max(ax1, bx1), max(ay1, by1), min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih
    ua = (ax2 - ax1) * (ay2 - ay1) + (bx2 - bx1) * (by2 - by1) - inter
    return inter / ua if ua > 0 else 0


kept = []
for c in range(5):
    cand = sorted([d for d in dets if d[0] == c], key=lambda d: -d[5])
    while cand:
        best = cand.pop(0)
        kept.append(best)
        cand = [d for d in cand if iou(best, d) <= iou_thr]

calib = {}
for cls, cx, cy, w, h, conf in kept:
    ox, oy = (cx - padx) / scale, (cy - pady) / scale
    nx, ny = ox / w0, oy / h0
    print(f"class={cls} conf={conf:.3f} norm=({nx:.3f},{ny:.3f})")
    if cls in (1, 2, 3, 4):
        calib[cls] = (nx, ny)

if len(calib) == 4:
    cx0 = sum(p[0] for p in calib.values()) / 4
    cy0 = sum(p[1] for p in calib.values()) / 4
    r = sum(math.hypot(p[0] - cx0, p[1] - cy0) for p in calib.values()) / 4
    # Canonical angles derived from the DeepDarts reference implementation's annotate.py
    # (transform(angle=9) template) and confirmed empirically against this dataset's own
    # labels -- see CalibrationPointMapper.kt / research.md §3.
    canon = {1: -9, 2: 171, 3: 261, 4: 81}
    sx = sy = 0.0
    for cls, (x, y) in calib.items():
        dx, dy = x - cx0, y - cy0
        ang = math.degrees(math.atan2(dx, -dy))
        offset = ang - canon[cls]
        sx += math.sin(math.radians(offset))
        sy += math.cos(math.radians(offset))
    rot = math.degrees(math.atan2(sx, sy))
    print(f"center=({cx0:.3f},{cy0:.3f}) r={r:.3f} rotationOffset={rot:.1f}")
else:
    print("WARNING: did not detect all 4 calibration points:", calib.keys())